In [109]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [110]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# Preprocessing
right shift -> tokeination -> indices

In [111]:
def preprocess(sentence):
    shift = "<sos> "+sentence
    
    sentence = shift.lower()
    tokens = sentence.split()
    print("Tokens : ",tokens)
    
    vocab = {word : idx for idx,word in enumerate(tokens)}

    #convert tokens to incides
    indices = torch.tensor([vocab[word] for word in tokens])

    return vocab,indices

# Embedding

In [112]:
def Embedding(vocab,indices,d_model) :
    embedding = nn.Embedding(num_embeddings=len(vocab), embedding_dim = d_model)
    x = embedding(indices)
    x = x.unsqueeze(0)
    return x

# Positional Encoding

In [113]:
class PositionalEncoding(nn.Module):
    def __init__(self, seq_len, d_model, max_len=1000):
        super().__init__()

        pos_encoding = torch.zeros(seq_len,d_model)

        position = torch.arange(seq_len).unsqueeze(1).float()
        dimension = torch.arange(d_model).unsqueeze(0).float()

        #angle rate
        angle_rate = 1 / (10000**((2*(dimension//2)) / d_model))

        #angle in radians
        angle_radians = position * angle_rate

        #sin and cos for even and odd pos
        pos_encoding[:,0::2] = torch.sin(angle_radians[:,0::2])
        pos_encoding[:,1::2] = torch.cos(angle_radians[:,1::2])

        #store encoding to register
        self.register_buffer("pos_encoding", pos_encoding)

    def forward(self,x):
        return x + self.pos_encoding.unsqueeze(0)

# Masked Multihead Attention

In [114]:
class MaskedMultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # learnable weight matrices
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def forward(self, x):
        batch, seq_len, _ = x.shape

        # Linear projections
        Q = self.W_Q(x)
        K = self.W_K(x)
        V = self.W_V(x)

        # Split into heads
        Q = Q.view(batch, seq_len, self.num_heads, self.d_k).permute(0,2,1,3)
        K = K.view(batch, seq_len, self.num_heads, self.d_k).permute(0,2,1,3)
        V = V.view(batch, seq_len, self.num_heads, self.d_k).permute(0,2,1,3)

        # Scaled dot product
        scores = (Q @ K.transpose(-2, -1)) / (self.d_k ** 0.5)

        # Causal mask
        mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device), diagonal=1)
        scores = scores.masked_fill(mask == 1, -1e9)

        # Softmax
        attention_weights = F.softmax(scores, dim=-1)

        # Attention output
        output = attention_weights @ V

        # Combine heads
        output = output.permute(0,2,1,3).contiguous()
        output = output.view(batch, seq_len, self.d_model)

        return self.W_O(output)

# Residual & Normalization Layer

In [115]:
class AddNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)

    def forward(self, a, b):
        return self.norm(a + b)

# Cross Attention

In [116]:
class CrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # Learnable projections
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def forward(self, dec_out, enc_out):
        batch, T_dec, _ = dec_out.shape
        _, T_enc, _ = enc_out.shape

        # Linear projections
        Q = self.W_Q(dec_out)   # (B, T_dec, d_model)
        K = self.W_K(enc_out)   # (B, T_enc, d_model)
        V = self.W_V(enc_out)   # (B, T_enc, d_model)

        # Split heads
        Q = Q.view(batch, T_dec, self.num_heads, self.d_k).permute(0,2,1,3)
        K = K.view(batch, T_enc, self.num_heads, self.d_k).permute(0,2,1,3)
        V = V.view(batch, T_enc, self.num_heads, self.d_k).permute(0,2,1,3)

        # Scaled dot product
        scores = (Q @ K.transpose(-2, -1)) / (self.d_k ** 0.5)
        attn = F.softmax(scores, dim=-1)

        # Attention output
        out = attn @ V

        # Combine heads
        out = out.permute(0,2,1,3).contiguous()
        out = out.view(batch, T_dec, self.d_model)

        return self.W_O(out)

# Feed Forward

In [117]:
class FeedForward(nn.Module):
    def __init__(self,d_model,d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model,d_ff)
        self.linear2 = nn.Linear(d_ff,d_model)
        self.relu = nn.ReLU()

    def forward(self,x):
        return self.linear2(self.relu(self.linear1(x)))

# Decoding an input sentence

In [118]:
d_model = 512
num_heads = 2
d_ff = 2048

In [119]:
sentence = 'Hello I am Manthan'
print("Input : ",sentence)

vocab,indices = preprocess(sentence)
print("Indices : ",indices)

x = Embedding(vocab,indices,d_model)
print("\nAfter converting to embedding : \n",x)
print("Shape : ",x.shape)

Input :  Hello I am Manthan
Tokens :  ['<sos>', 'hello', 'i', 'am', 'manthan']
Indices :  tensor([0, 1, 2, 3, 4])

After converting to embedding : 
 tensor([[[-1.5293,  0.0046, -0.4173,  ...,  0.6094,  0.0696, -2.7837],
         [ 1.4128,  0.6478,  0.5271,  ...,  1.0173,  0.2077,  0.3647],
         [ 1.8287, -1.8732, -0.3834,  ...,  0.5562,  0.7165, -2.0615],
         [-0.1828, -1.0307,  0.6350,  ...,  0.6574,  0.3192, -0.4023],
         [ 0.3993, -2.1132,  0.7776,  ..., -0.5908,  0.4150, -1.1249]]],
       grad_fn=<UnsqueezeBackward0>)
Shape :  torch.Size([1, 5, 512])


In [120]:
pos_encoded = PositionalEncoding(len(vocab),d_model)
x = pos_encoded(x)

print(x)
print(x.shape)

tensor([[[-1.5293,  1.0046, -0.4173,  ...,  1.6094,  0.0696, -1.7837],
         [ 2.2542,  1.1881,  1.3489,  ...,  2.0173,  0.2078,  1.3647],
         [ 2.7379, -2.2893,  0.5530,  ...,  1.5562,  0.7168, -1.0615],
         [-0.0417, -2.0207,  0.8801,  ...,  1.6574,  0.3195,  0.5977],
         [-0.3575, -2.7669,  0.1205,  ...,  0.4092,  0.4154, -0.1249]]],
       grad_fn=<AddBackward0>)
torch.Size([1, 5, 512])


In [121]:
#masked multi head attention
masked_mha = MaskedMultiHeadAttention(d_model,num_heads)
z = masked_mha(x)
print("After masked multi head attention : \n",z)
print("Shape : ",z.shape)

After masked multi head attention : 
 tensor([[[-0.0809, -0.0351, -0.3817,  ...,  0.0315, -0.7961, -0.0943],
         [-0.1770, -0.0147, -0.3251,  ...,  0.1450, -0.3981, -0.1131],
         [-0.1028,  0.0670, -0.0425,  ...,  0.0256, -0.3935, -0.1627],
         [ 0.0027,  0.0732, -0.1376,  ...,  0.1698, -0.3236, -0.1847],
         [-0.0592,  0.1242, -0.0181,  ...,  0.0241, -0.3461, -0.2366]]],
       grad_fn=<ViewBackward0>)
Shape :  torch.Size([1, 5, 512])


In [122]:
#residual and normalization
add_norm = AddNorm(d_model)
z_norm = add_norm(x,z)
print("After residual and normalization : \n", z_norm)
print(z_norm.shape)

After residual and normalization : 
 tensor([[[-1.8136,  0.4015, -1.1171,  ...,  0.9780, -1.0548, -2.0436],
         [ 1.3606,  0.5575,  0.4247,  ...,  1.4362, -0.6542,  0.6270],
         [ 1.9696, -2.5679, -0.0150,  ...,  0.9857, -0.1900, -1.6356],
         [-0.4925, -2.1831,  0.1997,  ...,  1.1607, -0.4617, -0.0921],
         [-0.7854, -2.6733, -0.3451,  ..., -0.0644, -0.3731, -0.7385]]],
       grad_fn=<NativeLayerNormBackward0>)
torch.Size([1, 5, 512])


In [123]:
#cross attention

#as of now, value of enc_out is kept random
enc_output = torch.randn(1,7,512)

cross_attention = CrossAttention(d_model,num_heads)
zc = cross_attention(z_norm,enc_output)
print("After cross attention : \n",zc)
print(zc.shape)

After cross attention : 
 tensor([[[ 0.0861, -0.1827, -0.1038,  ...,  0.1666, -0.1478,  0.0032],
         [ 0.1476, -0.1755, -0.0959,  ...,  0.1158, -0.1806,  0.1025],
         [ 0.1042, -0.2114, -0.0593,  ...,  0.2148, -0.0987,  0.1420],
         [ 0.1293, -0.1718, -0.0889,  ...,  0.1373, -0.1268,  0.1512],
         [ 0.1156, -0.1593, -0.0784,  ...,  0.1139, -0.0693,  0.1162]]],
       grad_fn=<ViewBackward0>)
torch.Size([1, 5, 512])


In [124]:
#add & norm
add_norm = AddNorm(d_model)
zc_norm = add_norm(z_norm,zc)
print("After residual and normalization : \n", zc_norm)
print(zc_norm.shape)

After residual and normalization : 
 tensor([[[-1.7193,  0.2136, -1.2162,  ...,  1.1330, -1.1980, -2.0301],
         [ 1.4907,  0.3695,  0.3164,  ...,  1.5343, -0.8421,  0.7153],
         [ 2.0787, -2.8029, -0.0820,  ...,  1.2002, -0.2978, -1.5097],
         [-0.3714, -2.3624,  0.1025,  ...,  1.2893, -0.5965,  0.0508],
         [-0.6709, -2.8161, -0.4266,  ...,  0.0426, -0.4453, -0.6238]]],
       grad_fn=<NativeLayerNormBackward0>)
torch.Size([1, 5, 512])


In [125]:
#feed forward
ff = FeedForward(d_model,d_ff=2048)
ff_output = ff(zc_norm)
print("After feed forward : \n",ff_output)
print(ff_output.shape)

After feed forward : 
 tensor([[[ 0.3478,  0.2362, -0.3003,  ...,  0.1387,  0.1259, -0.2342],
         [ 0.2771,  0.0993,  0.1025,  ...,  0.0485,  0.0030, -0.0771],
         [ 0.2260,  0.0893, -0.1063,  ..., -0.1295,  0.2689, -0.0177],
         [ 0.3193,  0.1496, -0.1216,  ...,  0.0235,  0.1805, -0.1881],
         [ 0.1293, -0.1061,  0.0771,  ..., -0.0745,  0.4010, -0.1862]]],
       grad_fn=<ViewBackward0>)
torch.Size([1, 5, 512])


In [126]:
#add & norm
add_norm = AddNorm(d_model)
y = add_norm(zc_norm,ff_output)
print("After residual and normalization : \n", y)
print(y.shape)

After residual and normalization : 
 tensor([[[-1.3115,  0.4412, -1.4510,  ...,  1.2321, -1.0233, -2.1707],
         [ 1.7239,  0.4567,  0.4080,  ...,  1.5434, -0.8193,  0.6220],
         [ 2.2277, -2.6096, -0.1755,  ...,  1.0382, -0.0218, -1.4663],
         [-0.0360, -2.1143, -0.0043,  ...,  1.2768, -0.3861, -0.1179],
         [-0.5218, -2.8261, -0.3358,  ..., -0.0285, -0.0404, -0.7816]]],
       grad_fn=<NativeLayerNormBackward0>)
torch.Size([1, 5, 512])


# **Complete Decoder Block**

In [127]:
class DecoderBlock(nn.Module):
    def __init__(self,d_model,num_heads,d_ff):
        super().__init__()
        self.masked_mha = MaskedMultiHeadAttention(d_model,num_heads)
        self.cross_attn = CrossAttention(d_model,num_heads)
        self.fnn = FeedForward(d_model,d_ff)

        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.ln3 = nn.LayerNorm(d_model)

    def forward(self,x,enc_output):
        z1 = self.masked_mha(x)
        print('-'*80)
        print("\tAfter Masked Multi Head Attention")
        print('-'*80)
        print(z1)
        print(z1.shape)
        
        z1_norm = self.ln1(x+z1)

        z2 = self.cross_attn(z1_norm,enc_output)
        print('-'*80)
        print("\tAfter Cross Attention")
        print('-'*80)
        print(z2)
        print(z2.shape)
        
        z2_norm = self.ln2(z1_norm+z2)

        z3 = self.fnn(z2_norm)
        print('-'*80)
        print("\tAfter Feed Forward")
        print('-'*80)
        print(z3)
        print(z3.shape)
        
        output = self.ln2(z2_norm+z3)

        return output

In [128]:
d_model = 512
num_heads = 2
d_ff = 2048

sentence = 'Hello my name is Manthan'
print("Input : ",sentence)

vocab,indices = preprocess(sentence)
print("Indices : ",indices)

x = Embedding(vocab,indices,d_model)
print('-'*80)
print("\tAfter converting to embedding")
print('-'*80)
print(x)
print("Shape : ",x.shape)

#as of now, value of enc_out is kept random
enc_output = torch.randn(1,9,512)
print('-'*80)
print("\tRandom output of encoder")
print('-'*80)
print(enc_output)
print(enc_output.shape)

decoder = DecoderBlock(d_model,num_heads,d_ff)
final_output = decoder(x,enc_output)
print('-'*80)
print("\tAfter passing through decoder block")
print('-'*80)
print(final_output)
print(final_output.shape)

Input :  Hello my name is Manthan
Tokens :  ['<sos>', 'hello', 'my', 'name', 'is', 'manthan']
Indices :  tensor([0, 1, 2, 3, 4, 5])
--------------------------------------------------------------------------------
	After converting to embedding
--------------------------------------------------------------------------------
tensor([[[-1.5535, -0.5678, -1.4876,  ..., -1.6529, -0.9819,  1.0604],
         [ 0.3866, -0.7194,  1.0378,  ..., -1.9136, -1.0687,  0.7902],
         [-0.8407,  0.1116, -1.5586,  ...,  0.1206,  0.2107, -0.3264],
         [ 0.2379,  0.4969, -0.0746,  ...,  0.6988,  0.8198, -0.0456],
         [ 0.7063, -0.5374,  1.0221,  ...,  0.2153, -1.2559, -0.3782],
         [-0.8996, -3.2424, -0.5411,  ...,  0.1703, -0.4851,  0.6063]]],
       grad_fn=<UnsqueezeBackward0>)
Shape :  torch.Size([1, 6, 512])
--------------------------------------------------------------------------------
	Random output of encoder
----------------------------------------------------------------------